# PoC - Modelo de Pressao Regional de Mobilidade

Este notebook cria um modelo simples para o servico `regional-mobility-pressure` usando duas fontes:
- `traffic-flow`
- `bus-regional-supply`

Saidas do modelo:
- classe: `baixa`, `media`, `alta`
- indice: `pressure_index` (0 a 100)


In [6]:
from pathlib import Path
import json
import numpy as np
import pandas as pd
import joblib

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

TRAFFIC_PATH = Path('../../L0/traffic-flow/output-example.txt').resolve()
BUS_SUPPLY_PATH = Path('../../L0/bus-regional-supply/output-example.txt').resolve()
MODEL_PATH = Path('model.joblib').resolve()

TRAFFIC_PATH, BUS_SUPPLY_PATH, MODEL_PATH


(PosixPath('/home/makleyston/Projects/service-composition/services/L0/traffic-flow/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L0/bus-regional-supply/output-example.txt'),
 PosixPath('/home/makleyston/Projects/service-composition/services/L1/regional-mobility-pressure/model.joblib'))

In [7]:
def load_jsonl(path: Path) -> list[dict]:
    records = []
    with path.open('r', encoding='utf-8') as f:
        for line in f:
            line = line.strip()
            if not line:
                continue
            records.append(json.loads(line))
    return records


def parse_traffic(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        payload = rec.get('urn:ufcity:payload', {})
        features = payload.get('features', {})
        result = rec.get('saref:hasResult', {})
        rows.append(
            {
                'traffic_level': result.get('saref:hasValue'),
                'traffic_confidence': result.get('urn:ufcity:confidence'),
                'traffic_id_eqp': features.get('ID EQP'),
                'traffic_direction': features.get('SENTIDO'),
                'traffic_lane': features.get('FAIXA'),
                'vehicle_count': features.get('vehicle_count'),
                'avg_speed': features.get('avg_speed'),
                'speed_std': features.get('speed_std'),
                'pct_moto': features.get('pct_moto'),
                'pct_heavy': features.get('pct_heavy'),
                'traffic_hour': features.get('hour'),
                'traffic_weekday': features.get('weekday'),
            }
        )
    return pd.DataFrame(rows)


def parse_bus_supply(records: list[dict]) -> pd.DataFrame:
    rows = []
    for rec in records:
        payload = rec.get('urn:ufcity:payload', {})
        result = rec.get('saref:hasResult', {})
        hr = str(payload.get('HR', ''))
        hour = int(hr[8:10]) if len(hr) >= 10 else np.nan
        weekday = pd.to_datetime(hr, format='%Y%m%d%H%M%S', errors='coerce').dayofweek if len(hr) == 14 else np.nan
        rows.append(
            {
                'bus_supply_level': result.get('saref:hasValue'),
                'bus_supply_confidence': result.get('urn:ufcity:confidence'),
                'bus_ev': payload.get('EV'),
                'bus_hr': payload.get('HR'),
                'bus_lt': payload.get('LT'),
                'bus_lg': payload.get('LG'),
                'bus_nv': payload.get('NV'),
                'bus_vl': payload.get('VL'),
                'bus_nl': payload.get('NL'),
                'bus_dg': payload.get('DG'),
                'bus_sv': payload.get('SV'),
                'bus_dt': payload.get('DT'),
                'bus_hour': hour,
                'bus_weekday': weekday,
            }
        )
    return pd.DataFrame(rows)


traffic_df = parse_traffic(load_jsonl(TRAFFIC_PATH))
bus_df = parse_bus_supply(load_jsonl(BUS_SUPPLY_PATH))

traffic_df.shape, bus_df.shape


((5819, 12), (3282, 14))

In [8]:
# Combinacao sintetica para PoC (nao existe alinhamento temporal real entre as fontes)
rng = np.random.default_rng(42)
sample_size = min(len(traffic_df), len(bus_df))

traffic_sample = traffic_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)
bus_sample = bus_df.sample(n=sample_size, random_state=42, replace=False).reset_index(drop=True)

df = pd.concat([traffic_sample, bus_sample], axis=1)

# Normalizacao simples de tipos
numeric_cols = [
    'traffic_confidence', 'traffic_id_eqp', 'traffic_lane', 'vehicle_count', 'avg_speed', 'speed_std',
    'pct_moto', 'pct_heavy', 'traffic_hour', 'traffic_weekday',
    'bus_supply_confidence', 'bus_ev', 'bus_hr', 'bus_lt', 'bus_lg', 'bus_nv',
    'bus_vl', 'bus_nl', 'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]
for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors='coerce')

traffic_map = {'baixo': 20, 'moderado': 60, 'alto': 90}
supply_map = {'alta': 20, 'media': 50, 'baixa': 80}

traffic_score = df['traffic_level'].map(traffic_map).fillna(50)
supply_score = df['bus_supply_level'].map(supply_map).fillna(50)
speed_penalty = np.clip((50 - df['avg_speed'].fillna(30)) * 2, 0, 100)
delay_penalty = np.clip((df['bus_dt'].fillna(0) / 20000) * 100, 0, 100)

rule_score = (0.45 * traffic_score + 0.25 * supply_score + 0.20 * speed_penalty + 0.10 * delay_penalty) * 1.7
df['pressure_rule_score'] = np.clip(rule_score, 0, 100)

df['regional_mobility_pressure'] = pd.cut(
    df['pressure_rule_score'],
    bins=[-1, 40, 70, 101],
    labels=['baixa', 'media', 'alta'],
).astype('object')

df['regional_mobility_pressure'].value_counts(normalize=True).rename('proporcao')


regional_mobility_pressure
baixa    0.755027
media    0.230957
alta     0.014016
Name: proporcao, dtype: float64

In [9]:
feature_cols = [
    'traffic_level', 'traffic_confidence', 'traffic_direction', 'traffic_lane', 'vehicle_count',
    'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy', 'traffic_hour', 'traffic_weekday',
    'bus_supply_level', 'bus_supply_confidence', 'bus_lt', 'bus_lg', 'bus_vl', 'bus_nl',
    'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]
target_col = 'regional_mobility_pressure'

X = df[feature_cols]
y = df[target_col]

numeric_features = [
    'traffic_confidence', 'traffic_lane', 'vehicle_count', 'avg_speed', 'speed_std', 'pct_moto', 'pct_heavy',
    'traffic_hour', 'traffic_weekday', 'bus_supply_confidence', 'bus_lt', 'bus_lg', 'bus_vl', 'bus_nl',
    'bus_dg', 'bus_sv', 'bus_dt', 'bus_hour', 'bus_weekday'
]
categorical_features = ['traffic_level', 'traffic_direction', 'bus_supply_level']

numeric_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='median')),
        ('scaler', StandardScaler()),
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ('imputer', SimpleImputer(strategy='most_frequent')),
        ('onehot', OneHotEncoder(handle_unknown='ignore')),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ('num', numeric_transformer, numeric_features),
        ('cat', categorical_transformer, categorical_features),
    ]
)

clf = Pipeline(
    steps=[
        ('preprocessor', preprocessor),
        (
            'model',
            RandomForestClassifier(
                n_estimators=300,
                random_state=42,
                class_weight='balanced',
                n_jobs=-1,
            ),
        ),
    ]
)

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y,
)

clf.fit(X_train, y_train)
pred = clf.predict(X_test)

print(classification_report(y_test, pred))
print(confusion_matrix(y_test, pred))


              precision    recall  f1-score   support

        alta       1.00      1.00      1.00         9
       baixa       0.98      1.00      0.99       496
       media       1.00      0.94      0.97       152

    accuracy                           0.99       657
   macro avg       0.99      0.98      0.99       657
weighted avg       0.99      0.99      0.99       657

[[  9   0   0]
 [  0 496   0]
 [  0   9 143]]


In [10]:
index_weights = {'baixa': 0.0, 'media': 50.0, 'alta': 100.0}
classes = list(clf.named_steps['model'].classes_)
class_weight_vector = np.array([index_weights.get(c, 50.0) for c in classes], dtype=float)

proba = clf.predict_proba(X_test.head(10))
pressure_index = np.clip(np.dot(proba, class_weight_vector), 0, 100)

preview = X_test.head(10).copy()
preview['pressao_real'] = y_test.head(10).values
preview['pressao_predita'] = clf.predict(X_test.head(10))
preview['pressure_index'] = np.round(pressure_index, 1)
preview


,traffic_level,traffic_confidence,traffic_direction,traffic_lane,vehicle_count,avg_speed,speed_std,pct_moto,pct_heavy,traffic_hour,...,bus_vl,bus_nl,bus_dg,bus_sv,bus_dt,bus_hour,bus_weekday,pressao_real,pressao_predita,pressure_index
24,baixo,0.920,Betânia/Barreiro,2,25,49.200000,5.400617,0.080000,0.040000,0,...,0,6018.0,0,0.0,5498.0,17,5,baixa,baixa,0.3
1789,baixo,1.000,Centro/Bairro,1,2,51.000000,5.656854,0.500000,0.000000,0,...,12,768.0,188,1.0,798.0,17,5,baixa,baixa,1.5
560,baixo,1.000,Bairro/Centro,2,2,49.500000,0.707107,0.000000,0.000000,0,...,0,7419.0,189,0.0,11638.0,17,5,baixa,baixa,0.5
2155,baixo,0.940,Bairro/Centro,2,28,51.178571,4.784648,0.071429,0.000000,0,...,1,68.0,140,0.0,4026.0,17,5,baixa,baixa,0.0
2962,moderado,0.870,Centro/Bairro,2,89,51.393258,4.576665,0.191011,0.011236,0,...,0,803.0,0,0.0,6490.0,17,5,media,media,49.5
2078,baixo,0.985,Centro/Barro Preto,1,10,47.000000,7.133645,0.000000,0.100000,0,...,0,6844.0,352,0.0,0.0,17,5,baixa,baixa,1.0
3191,baixo,0.950,Praça da Estação / Rodoviária,1,7,43.285714,6.317022,0.000000,0.142857,0,...,0,597.0,146,1.0,21774.0,17,5,media,media,39.3
2976,baixo,1.000,Bairro/Centro,3,2,46.500000,19.091883,0.000000,1.000000,0,...,0,5784.0,213,0.0,3235.0,17,5,baixa,baixa,2.0
3127,baixo,0.905,Centro/Bairro,2,21,47.857143,5.198901,0.285714,0.047619,0,...,0,585.0,237,0.0,0.0,17,5,baixa,baixa,0.8
1718,baixo,0.720,Centro/Bairro,2,48,52.270833,5.018008,0.104167,0.000000,0,...,0,919.0,0,NaN,3142.0,17,5,baixa,baixa,0.7


In [11]:
artifact = {
    'pipeline': clf,
    'features': feature_cols,
    'target': target_col,
    'classes': classes,
    'index_name': 'pressure_index',
    'index_scale': [0, 100],
    'index_weights': index_weights,
    'labeling_rule': {
        'score_formula': '(0.45*traffic + 0.25*supply + 0.20*speed_penalty + 0.10*delay_penalty) * 1.7',
        'baixa': 'score <= 40',
        'media': '40 < score <= 70',
        'alta': 'score > 70',
    },
}

joblib.dump(artifact, MODEL_PATH)
MODEL_PATH


PosixPath('/home/makleyston/Projects/service-composition/services/L1/regional-mobility-pressure/model.joblib')